# Aligned feature extraction - 12 카테고리 (~241 features)

01-registration 에서 정렬된 mask 를 기반으로 12 카테고리 feature 추출:

**기존 6 카테고리 (188)**:
| Part | Count |
|---|---|
| Whole density 7 + fractal 3 + sholl 24 | 34 |
| Zone (11 x 7) | 77 |
| Skeleton (whole 9 + segments 15) | 24 |
| Caliber (3 net x 6 stat) | 18 |
| Knudtson CRAE/CRVE/AVR + n_seg | 5 |
| Color (3 net x 9 + AV 3) | 30 |

**신규 6 카테고리 (+53)**:
| Part | Count | 임상 |
|---|---|---|
| Tortuosity | 3 net x 4 stat = 12 | 고혈압, DR 초기, ROP |
| FAZ (Foveal Avascular Zone) | 5 | DR/DME/ischemia gold standard |
| Crossings (AV nicking) | 3 area x 3 stat = 9 | 고혈압성 망막병증 |
| Bifurcation angle | 3 net x 4 stat = 12 | Vascular disease (Murray's law) |
| Width variability | 3 net x 2 stat = 6 | Focal narrowing (동맥경화) |
| Junction exponent | 3 net x 3 stat = 9 | Murray junction X (정상 ~3) |

**TOTAL ~241 features (+7 meta = ~248 columns)**

각 파트마다: **추출 함수 → extraction 코드 → timeline plot**.


## 0. Setup — imports, paths, inventory, zones, common_valid


In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

PKG_ROOT = Path("..").resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from features.zones import (
    make_etdrs_9subfields, make_disc_zones, mm_per_px_from_disc,
    ETDRS_9_NAMES, DISC_ZONE_NAMES,
    DEFAULT_ETDRS_RADII_MM, DEFAULT_DISC_MARGIN_ZONES, STANDARD_DISC_MM,
)
from features.density  import zone_density_features, whole_image_density_features
from features.skeleton import (
    zone_skeleton_features, whole_image_skeleton_features, whole_image_segment_features,
    make_skeletons, clean_and_skeletonize, branch_endpoint_masks,
    whole_image_tortuosity_features, whole_image_bifurcation_features,
)
from features.caliber  import (
    zone_caliber_features, whole_image_caliber_features,
    compute_crae_crve_avr, compute_caliber_map,
    whole_image_width_variability, whole_image_junction_features,
)
from features.topology import whole_image_fractal_features, sholl_features, sholl_curve
from features.color    import whole_image_color_features, color_stats_at_skeleton
from features.faz      import compute_faz
from features.crossings import crossings_features


In [ ]:
SQUARE_SIZE = 512
src_name    = "20230118_HCS_Fundus_image"

rgb_path     = Path(f"g:/RAG/project_1/output/preprocessed_{SQUARE_SIZE}") / src_name / "rgb"
aligned_root = Path(f"g:/RAG/project_1/output/aligned_{SQUARE_SIZE}") / src_name
features_out = Path(f"g:/RAG/project_1/output/features_json_aligned_{SQUARE_SIZE}") / src_name
features_out.mkdir(parents=True, exist_ok=True)

inv_df   = pd.read_csv(aligned_root / "inventory.csv", parse_dates=["date"])
reg_log  = pd.read_csv(aligned_root / "registration_log.csv", parse_dates=["date"])

inventory = {}
for (pid, eye), grp in inv_df.groupby(["patient", "eye"], sort=False):
    inventory.setdefault(pid, {})[eye] = grp.sort_values(["date", "shot"]).to_dict("records")

theta_cols = [f"t{i}{j}" for i in range(3) for j in range(3)]
def get_theta(image_id):
    row = reg_log[reg_log["id"] == image_id]
    if len(row) == 0: return None
    return row.iloc[0][theta_cols].values.astype(np.float64).reshape(3, 3)

def warp_image(img, theta, H, W, is_mask=False):
    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    return cv2.warpAffine(img, theta[:2, :], (W, H),
                          flags=flags, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

patient_ids = sorted(inventory.keys())
for pid in patient_ids:
    for eye, visits in inventory[pid].items():
        print(f"{pid}/{eye}: {len(visits)} visits")


In [ ]:
# Zones per (patient, eye) — ETDRS 9 + Zone B/C (fixed visit reference)
zones_per_group = {}
zone_meta = []
for pid, by_eye in inventory.items():
    for eye, visits in by_eye.items():
        if eye == "unknown" or len(visits) < 1: continue
        fixed = visits[0]
        fx, fy = fixed["x_fovea"], fixed["y_fovea"]
        dcx, dcy = fixed["x_disc"], fixed["y_disc"]
        grp_df = inv_df[(inv_df["patient"]==pid) & (inv_df["eye"]==eye)]
        dd_px = grp_df["disc_diameter"].mean()
        mpp = mm_per_px_from_disc(dd_px)
        etdrs  = make_etdrs_9subfields(fx, fy, eye, mpp, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_ETDRS_RADII_MM)
        disc_z = make_disc_zones(dcx, dcy, dd_px, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_DISC_MARGIN_ZONES)
        zones = {**etdrs, **disc_z}
        zones_per_group[(pid, eye)] = zones
        zone_meta.append({
            "patient": pid, "eye": eye,
            "fovea_x": fx, "fovea_y": fy,
            "disc_cx": dcx, "disc_cy": dcy,
            "disc_diameter_px": dd_px, "mm_per_px": mpp,
        })
zone_meta_df = pd.DataFrame(zone_meta)
zone_meta_df


In [ ]:
# common_valid per group — 모든 visit 의 fundus 유효 영역 교집합
def fundus_valid(rgb): return rgb.sum(-1) > 10

def compute_common_valid(visits):
    common = np.ones((SQUARE_SIZE, SQUARE_SIZE), dtype=bool)
    for i, v in enumerate(visits):
        rgb = np.array(Image.open(rgb_path / f"{v['id']}.png"))
        if i == 0:
            common &= fundus_valid(rgb)
        else:
            theta = get_theta(v["id"])
            if theta is None: continue
            common &= fundus_valid(warp_image(rgb, theta, SQUARE_SIZE, SQUARE_SIZE))
    return common

common_valid_per_group = {
    key: compute_common_valid(inventory[key[0]][key[1]])
    for key in zones_per_group.keys()
}
print("common_valid 계산 완료")


## Extraction — 모든 카테고리 한 번에 (single loop)


In [ ]:
feat_rows = []
for pid, by_eye in tqdm(inventory.items(), desc="patients"):
    for eye, visits in by_eye.items():
        if eye == "unknown": continue

        key = (pid, eye)
        if key not in zones_per_group: continue
        zones    = zones_per_group[key]
        valid    = common_valid_per_group[key]
        meta     = zone_meta_df.query("patient==@pid & eye==@eye").iloc[0]
        mpp_grp  = meta["mm_per_px"]
        dd_grp   = meta["disc_diameter_px"]
        fx, fy   = meta["fovea_x"], meta["fovea_y"]
        dcx, dcy = meta["disc_cx"], meta["disc_cy"]
        fixed_id = visits[0]["id"]
        aligned  = aligned_root / str(pid) / eye

        for v in visits:
            vid = v["id"]
            try:
                rgb_base = np.array(Image.open(rgb_path / f"{vid}.png"))
                vs_m = np.array(Image.open(aligned / "vessels" / f"{vid}.png"))
                av_m = np.array(Image.open(aligned / "av"      / f"{vid}.png"))
                dc_m = np.array(Image.open(aligned / "discs"   / f"{vid}.png"))
            except FileNotFoundError:
                continue

            dd_v = v.get("disc_diameter", np.nan)
            if np.isfinite(dd_v) and dd_v >= 5:
                mpp_v = mm_per_px_from_disc(dd_v)
            else:
                dd_v, mpp_v = float(dd_grp), float(mpp_grp)

            if vid == fixed_id:
                rgb_a = rgb_base
            else:
                theta = get_theta(vid)
                rgb_a = warp_image(rgb_base, theta, SQUARE_SIZE, SQUARE_SIZE) if theta is not None else rgb_base

            # ── Existing 6 카테고리 ──
            w_dens   = whole_image_density_features(vs_m, av_m, dc_m, valid_mask=valid)
            z_dens   = zone_density_features(vs_m, av_m, dc_m, zones, valid_mask=valid)
            w_skel   = whole_image_skeleton_features(vs_m, av_m, valid_mask=valid)
            w_seg    = whole_image_segment_features(vs_m, av_m, valid_mask=valid)
            w_cal    = whole_image_caliber_features(vs_m, av_m, valid_mask=valid, mm_per_px=mpp_v)
            knudtson = compute_crae_crve_avr(av_m, zones["zone_B"], mm_per_px=mpp_v)
            w_frac   = whole_image_fractal_features(vs_m, av_m, valid_mask=valid)
            w_sh_f   = sholl_features(vs_m, av_m, (fx, fy),   valid_mask=valid, center_name="fovea")
            w_sh_d   = sholl_features(vs_m, av_m, (dcx, dcy), valid_mask=valid, center_name="disc")
            w_color  = whole_image_color_features(rgb_a, vs_m, av_m, valid_mask=valid)

            # ── New Tier 1+2 ──
            w_tort   = whole_image_tortuosity_features(vs_m, av_m, valid_mask=valid)
            faz      = compute_faz(vs_m, (fx, fy), valid_mask=valid, mm_per_px=mpp_v, disc_diameter_px=dd_v)
            cross    = crossings_features(av_m, zones=zones, valid_mask=valid)
            w_bif    = whole_image_bifurcation_features(vs_m, av_m, valid_mask=valid)
            w_wv     = whole_image_width_variability(vs_m, av_m, valid_mask=valid)
            w_junc   = whole_image_junction_features(vs_m, av_m, valid_mask=valid)

            feat_rows.append({
                "id": vid, "patient": pid, "eye": eye,
                "date": v["date"], "shot": v["shot"],
                "disc_diameter_px": dd_v, "mm_per_px": mpp_v,
                **w_dens, **z_dens, **w_skel, **w_seg, **w_cal, **knudtson,
                **w_frac, **w_sh_f, **w_sh_d, **w_color,
                **w_tort, **faz, **cross, **w_bif, **w_wv, **w_junc,
            })

features_zones = pd.DataFrame(feat_rows).sort_values(["patient","eye","date"]).reset_index(drop=True)

# ── 환자별 JSON 저장 (mergeable to CSV) ──
import json as _json
from datetime import datetime as _dt

def _jd(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)):
        v = float(o); return None if not np.isfinite(v) else v
    if isinstance(o, np.ndarray):     return o.tolist()
    if isinstance(o, pd.Timestamp):   return o.isoformat()
    return str(o)

n_saved = 0
for pid, pdf in features_zones.groupby("patient", sort=False):
    eye_meta = {}
    for eye in pdf["eye"].unique():
        mrow = zone_meta_df.query("patient==@pid & eye==@eye")
        if len(mrow) == 0: continue
        m = mrow.iloc[0]
        eye_meta[eye] = {
            "fovea_x": float(m["fovea_x"]), "fovea_y": float(m["fovea_y"]),
            "disc_cx": float(m["disc_cx"]), "disc_cy": float(m["disc_cy"]),
            "disc_diameter_px": float(m["disc_diameter_px"]),
            "mm_per_px": float(m["mm_per_px"]),
            "n_visits": int((pdf["eye"] == eye).sum()),
        }
    payload = {
        "patient": str(pid),
        "config": {"mode": "aligned", "square_size": SQUARE_SIZE,
                   "extracted_at": _dt.utcnow().isoformat() + "Z"},
        "meta_by_eye": eye_meta,
        "features": pdf.assign(date=pdf["date"].astype(str)).to_dict("records"),
    }
    with open(features_out / f"{pid}.json", "w", encoding="utf-8") as f:
        _json.dump(payload, f, ensure_ascii=False, indent=2, default=_jd)
    n_saved += 1

# 편의용 merged CSV (downstream viz 셀 호환)
features_zones.to_csv(features_out / "_merged_features_aligned.csv", index=False)
print(f"저장: {n_saved} patient JSON → {features_out}")
print(f"       + _merged_features_aligned.csv (rows={len(features_zones)}, cols={features_zones.shape[1]})")
features_zones.head()


# 🌐 Part 1. Whole-image features (35)

**전체 fundus (`valid_mask` 안) 에서 계산** — zone 무관 global 지표.

| 서브카테고리 | 함수 | Count | 설명 |
|---|---|---|---|
| Density | `whole_image_density_features` | 8 | vessel/artery/vein density, AV_ratio, disc_frac + area counts |
| Fractal dim | `whole_image_fractal_features` | 3 | Box-counting fractal dimension (per network) |
| Sholl analysis | `sholl_features` × 2 centers | 24 | Fovea + disc 중심 동심원별 skeleton 교차 수 곡선 → 4 summary stat |


In [ ]:
# Whole density timeline (5 core metric)
whole_core_metrics = ["vessel_density", "artery_density", "vein_density", "AV_ratio_area", "disc_frac"]
fig, axes = plt.subplots(len(patient_ids), len(whole_core_metrics),
                         figsize=(4*len(whole_core_metrics), 3*len(patient_ids)), squeeze=False)
for row, pid in enumerate(patient_ids):
    for col, mb in enumerate(whole_core_metrics):
        ax = axes[row, col]
        col_name = f"whole_{mb}"
        for eye, style in [("L","-o"), ("R","-s")]:
            sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
            if len(sub) == 0 or col_name not in sub: continue
            ax.plot(sub["date"], sub[col_name], style, label=eye, markersize=3, linewidth=1)
        ax.set_title(f"{pid} — whole_{mb}", fontsize=9)
        ax.grid(alpha=0.3); ax.legend(fontsize=7)
        ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.suptitle("Part 1a — Whole-image density", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# Fractal dim timeline (3 networks per patient, one plot)
fig, axes = plt.subplots(1, len(patient_ids), figsize=(5*len(patient_ids), 4), squeeze=False)
for col, pid in enumerate(patient_ids):
    ax = axes[0, col]
    for net, color in [("vessel","tab:green"), ("artery","tab:red"), ("vein","tab:blue")]:
        col_name = f"whole_{net}_fractal_dim"
        for eye, ls in [("L","-"), ("R","--")]:
            sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
            if len(sub) == 0 or col_name not in sub: continue
            ax.plot(sub["date"], sub[col_name], ls+"o", color=color,
                    label=f"{eye} {net}", markersize=3, linewidth=1)
    ax.set_title(f"{pid} — Fractal dim (whole)", fontsize=10)
    ax.set_ylabel("D (fractal dim)"); ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=3); ax.tick_params(axis="x", rotation=45)
plt.suptitle("Part 1b — Fractal dimension", fontsize=13)
plt.tight_layout(); plt.show()


### 🎯 Sholl analysis 란?

**1953년 Sholl 이 신경세포 dendrite branching 분석용으로 개발** → retinal vessel network 에도 적용.

**원리**: 중심점 (fovea 또는 disc) 에서 반경 r 을 늘려가면서 각 원과 skeleton 이 몇 번 교차하는지 카운트 → r 별 crossing 수 곡선.

**추출 지표** (곡선 → 4 summary):
- `sholl_max` : 최대 crossing 수 (peak complexity)
- `sholl_r_max_px` : peak 반경 (vessel 가장 촘촘한 위치)
- `sholl_auc` : 곡선 아래 면적 (전체 복잡도)
- `sholl_mean` : 전 반경 평균

**두 중심을 다 쓰는 이유**:
- **fovea-centered** — Macula 주변, FAZ 크기 반영
- **disc-centered** — Peripapillary arcade 시작 지점 반영

**임상**: FAZ 확대 (DR 초기) → fovea Sholl 첫 crossing 반경 ↑, vessel attenuation → `sholl_max` ↓


In [ ]:
# Sholl summary timeline — 2 centers × 3 networks × 4 stats
sholl_stats = ["sholl_max", "sholl_r_max_px", "sholl_auc", "sholl_mean"]
for center in ["fovea", "disc"]:
    fig, axes = plt.subplots(3, len(sholl_stats),
                             figsize=(4*len(sholl_stats), 8), squeeze=False, sharex=True)
    networks = ["vessel", "artery", "vein"]
    for r, net in enumerate(networks):
        for c, st in enumerate(sholl_stats):
            ax = axes[r, c]
            col_name = f"sholl_{center}_{net}_{st}"
            for pid in patient_ids:
                for eye, style in [("L","-o"), ("R","-s")]:
                    sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
                    if len(sub) == 0 or col_name not in sub: continue
                    ax.plot(sub["date"], sub[col_name], style,
                            label=f"{pid} {eye}", markersize=3, linewidth=1)
            ax.set_title(f"{net} {st}", fontsize=9)
            ax.grid(alpha=0.3); ax.legend(fontsize=6, ncol=2)
            ax.tick_params(axis="x", rotation=45, labelsize=7)
    plt.suptitle(f"Part 1c — Sholl summary ({center}-centered)", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# Sholl actual curve — fixed visit per (patient, eye), single center 별
from features.topology import sholl_curve
for pid in patient_ids:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), squeeze=False)
    for col, center_name in enumerate(["fovea", "disc"]):
        ax = axes[0, col]
        radii = list(range(10, 240, 5))
        for eye in ["L", "R"]:
            if eye not in inventory[pid]: continue
            fixed = inventory[pid][eye][0]
            vs_m = np.array(Image.open(aligned_root / str(pid) / eye / "vessels" / f"{fixed['id']}.png"))
            skel = clean_and_skeletonize(vs_m > 0)
            meta = zone_meta_df.query("patient==@pid & eye==@eye").iloc[0]
            cx, cy = (meta["fovea_x"], meta["fovea_y"]) if center_name=="fovea" else (meta["disc_cx"], meta["disc_cy"])
            curve = sholl_curve(skel, cx, cy, radii)
            ax.plot(radii, curve, "-o", markersize=3, label=f"{eye} eye")
        ax.set_xlabel("radius (px)"); ax.set_ylabel("# skeleton crossings")
        ax.set_title(f"{pid} — Sholl curve ({center_name}-centered, fixed visit)")
        ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()


# 📍 Part 2. Zone-based density (88)

**11 zones × 8 metric** — 각 zone 안에서 vessel/artery/vein density + AV_ratio + disc_frac + area counts.

함수: `zone_density_features(vessel, av, disc, zones, valid_mask)`.

Zone 종류:
- ETDRS 9-subfield (fovea 중심): C, S1, N1, I1, T1, S2, N2, I2, T2
- Disc-margin (Wong): zone_B (0.5-1 DD), zone_C (1-2 DD)


In [ ]:
# Zone 시각화 — fixed 이미지 위에 ETDRS 9 + Zone B/C 오버레이
ZONE_COLORS = {
    # ETDRS 9-subfield (fovea-centered)
    "C":  [255, 255, 100],   "S1": [130, 200, 255], "N1": [180, 130, 255],
    "I1": [130, 255, 180],   "T1": [255, 180, 130], "S2": [80,  130, 200],
    "N2": [130, 80,  200],   "I2": [80,  200, 130], "T2": [200, 130, 80],
    # Disc-centered (Wong)
    "zone_B": [255, 100, 100],   # 밝은 빨강 — inner peripapillary
    "zone_C": [180, 60,  60],    # 어두운 빨강 — outer peripapillary
}

def overlay_zones(rgb, zones, alpha=0.4):
    disp = rgb.copy()
    for name, mask in zones.items():
        c = np.array(ZONE_COLORS.get(name, [200, 200, 200]))
        blend = (rgb[mask].astype(np.float32) * (1-alpha) + c * alpha)
        disp[mask] = blend.astype(np.uint8)
    return disp

n = len(zones_per_group)
fig, axes = plt.subplots(1, n, figsize=(5.5*n, 5.5))
if n == 1: axes = [axes]
for ax, ((pid, eye), zones) in zip(axes, zones_per_group.items()):
    fixed = inventory[pid][eye][0]
    rgb = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    ax.imshow(overlay_zones(rgb, zones))
    ax.scatter(fixed["x_fovea"], fixed["y_fovea"], c="white", marker="x",
               s=80, linewidths=2, label="fovea")
    ax.scatter(fixed["x_disc"],  fixed["y_disc"],  c="white", marker="o",
               s=80, linewidths=2, facecolors="none", label="disc")
    dd = zone_meta_df.query('patient==@pid & eye==@eye')['disc_diameter_px'].iloc[0]
    ax.set_title(f"{pid}/{eye} — ETDRS 9 + Zone B/C (DD={dd:.1f}px)")
    ax.axis("off"); ax.legend(loc="lower right", fontsize=8)

from matplotlib.patches import Patch
handles = [Patch(color=np.array(c)/255, label=n) for n, c in ZONE_COLORS.items()]
fig.legend(handles=handles, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 0.02), fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Zone density timeline — 3 metric × 11 zones per patient
zone_names = ETDRS_9_NAMES + DISC_ZONE_NAMES
core_metrics = ["vessel_density", "artery_density", "vein_density"]

for pid in patient_ids:
    fig, axes = plt.subplots(len(core_metrics), len(zone_names),
                             figsize=(2.4*len(zone_names), 2.4*len(core_metrics)),
                             squeeze=False, sharex=True)
    for r, mb in enumerate(core_metrics):
        for c, zn in enumerate(zone_names):
            ax = axes[r, c]
            col_name = f"{zn}_{mb}"
            for eye, style in [("L","-o"), ("R","-s")]:
                sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
                if len(sub) == 0 or col_name not in sub: continue
                ax.plot(sub["date"], sub[col_name], style, label=eye, markersize=2, linewidth=0.9)
            ax.set_title(f"{zn} {mb}", fontsize=7)
            ax.grid(alpha=0.3); ax.legend(fontsize=6)
            ax.tick_params(axis="x", rotation=45, labelsize=5)
    fig.suptitle(f"Patient {pid} — Part 2: Zone density (3 metric × 11 zones)", fontsize=13)
    plt.tight_layout(); plt.show()


# 🕸️ Part 3. Skeleton features (24)

**Whole-image skeleton 위에서** vessel/artery/vein 각각:

| 서브카테고리 | 함수 | Count |
|---|---|---|
| Skeleton (length, branches, endpoints) | `whole_image_skeleton_features` | 9 |
| Segment stats (n_segments, mean/median/max/std length) | `whole_image_segment_features` | 15 |

Segment = skeleton 에서 branch pixel 제거 후 connected component.


In [ ]:
# Skeleton overlay 시각화 — fixed 이미지 위
for (pid, eye), zones in zones_per_group.items():
    fixed = inventory[pid][eye][0]
    aligned = aligned_root / str(pid) / eye
    rgb  = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    vs_m = np.array(Image.open(aligned / "vessels" / f"{fixed['id']}.png"))
    av_m = np.array(Image.open(aligned / "av"      / f"{fixed['id']}.png"))
    skels = make_skeletons(vs_m, av_m)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, skel) in zip(axes, skels.items()):
        br, ep = branch_endpoint_masks(skel)
        disp = (rgb.astype(np.float32) * 0.35).astype(np.uint8)
        color = {"vessel":[0,255,60],"artery":[255,80,80],"vein":[80,120,255]}[name]
        disp[skel] = color; disp[br] = [255,255,0]; disp[ep] = [255,0,255]
        ax.imshow(disp)
        ax.set_title(f"{name} skeleton (yellow=branch, magenta=endpoint)")
        ax.axis("off")
    fig.suptitle(f"Patient {pid}/{eye} — skeleton (fixed visit)", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# Whole skeleton timeline — 3 network × 3 metric
skel_metrics = ["skel_length_px", "n_branches", "n_endpoints"]
networks = ["vessel", "artery", "vein"]
for pid in patient_ids:
    fig, axes = plt.subplots(3, 3, figsize=(14, 8), squeeze=False, sharex=True)
    for r, net in enumerate(networks):
        for c, sm in enumerate(skel_metrics):
            ax = axes[r, c]
            col_name = f"whole_{net}_{sm}"
            for eye, style in [("L","-o"), ("R","-s")]:
                sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
                if len(sub) == 0 or col_name not in sub: continue
                ax.plot(sub["date"], sub[col_name], style, label=eye, markersize=3, linewidth=1)
            ax.set_title(f"whole_{net}_{sm}", fontsize=9)
            ax.grid(alpha=0.3); ax.legend(fontsize=7)
            ax.tick_params(axis="x", rotation=45, labelsize=7)
    fig.suptitle(f"Patient {pid} — Part 3a: Skeleton whole (3 net × 3 metric)", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# Segment length stats timeline — 3 network × 5 stat
seg_stats = ["n_segments", "seg_len_mean", "seg_len_median", "seg_len_max", "seg_len_std"]
for pid in patient_ids:
    fig, axes = plt.subplots(3, 5, figsize=(20, 8), squeeze=False, sharex=True)
    for r, net in enumerate(networks):
        for c, st in enumerate(seg_stats):
            ax = axes[r, c]
            col_name = f"whole_{net}_{st}"
            for eye, style in [("L","-o"), ("R","-s")]:
                sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
                if len(sub) == 0 or col_name not in sub: continue
                ax.plot(sub["date"], sub[col_name], style, label=eye, markersize=3, linewidth=1)
            ax.set_title(f"{net} {st}", fontsize=9)
            ax.grid(alpha=0.3); ax.legend(fontsize=7)
            ax.tick_params(axis="x", rotation=45, labelsize=7)
    fig.suptitle(f"Patient {pid} — Part 3b: Segment length stats", fontsize=13)
    plt.tight_layout(); plt.show()


# 📏 Part 4. Caliber (18)

**Whole-image vessel width (μm)** — 각 skeleton pixel 의 diameter (distance transform × 2) 를 sampling.

함수: `whole_image_caliber_features(vessel, av, valid_mask, mm_per_px)`.

각 network (vessel/artery/vein) × 6 stat:
- `caliber_mean_um`, `caliber_median_um` — 평균 / 중앙 vessel 굵기
- `caliber_p75_um`, `caliber_p90_um` — 큰 vessel 대표
- `caliber_max_um` — 가장 큰 single vessel
- `caliber_n_samples` — skeleton pixel 수 (참고)


In [ ]:
# Caliber map 시각화 — skeleton 값을 vessel 전체로 전파
from scipy.ndimage import distance_transform_edt

def spread_caliber_to_vessel(vessel_bin, caliber_skel):
    if not vessel_bin.any() or not (caliber_skel > 0).any():
        return np.zeros_like(caliber_skel, dtype=np.float32)
    _, (ii, jj) = distance_transform_edt(~(caliber_skel > 0), return_indices=True)
    dense = caliber_skel[ii, jj].astype(np.float32)
    dense[~vessel_bin] = 0.0
    return dense

for (pid, eye), zones in zones_per_group.items():
    fixed = inventory[pid][eye][0]
    aligned = aligned_root / str(pid) / eye
    rgb  = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    vs_m = np.array(Image.open(aligned / "vessels" / f"{fixed['id']}.png"))
    av_m = np.array(Image.open(aligned / "av"      / f"{fixed['id']}.png"))
    mpp  = zone_meta_df.query("patient==@pid & eye==@eye")["mm_per_px"].iloc[0]

    cal_v = spread_caliber_to_vessel(vs_m > 0,               compute_caliber_map(vs_m > 0))
    cal_a = spread_caliber_to_vessel(np.isin(av_m,(1,3)),    compute_caliber_map(np.isin(av_m,(1,3))))
    cal_n = spread_caliber_to_vessel(np.isin(av_m,(2,3)),    compute_caliber_map(np.isin(av_m,(2,3))))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, cal, cmap) in zip(axes, [
        ("vessel", cal_v, "viridis"),
        ("artery", cal_a, "Reds"),
        ("vein",   cal_n, "Blues"),
    ]):
        disp = (rgb.astype(np.float32) * 0.35).astype(np.uint8)
        ax.imshow(disp)
        cal_um = cal * mpp * 1000.0
        vmax = float(np.percentile(cal_um[cal_um > 0], 99)) if (cal_um > 0).any() else 100
        im = ax.imshow(np.ma.masked_where(cal_um == 0, cal_um), cmap=cmap, alpha=1.0, vmin=0, vmax=vmax)
        plt.colorbar(im, ax=ax, fraction=0.04, label="μm")
        ax.set_title(f"{name} caliber (max={cal_um.max():.0f} μm)")
        ax.axis("off")
    fig.suptitle(f"Patient {pid}/{eye} — caliber map (fixed visit)", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# Whole caliber timeline — 3 network × 6 stat
cal_stats = ["caliber_mean_um","caliber_median_um","caliber_p75_um","caliber_p90_um","caliber_max_um","caliber_n_samples"]
for pid in patient_ids:
    fig, axes = plt.subplots(3, len(cal_stats),
                             figsize=(3.5*len(cal_stats), 8), squeeze=False, sharex=True)
    for r, net in enumerate(networks):
        for c, st in enumerate(cal_stats):
            ax = axes[r, c]
            col_name = f"whole_{net}_{st}"
            for eye, style in [("L","-o"), ("R","-s")]:
                sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
                if len(sub) == 0 or col_name not in sub: continue
                ax.plot(sub["date"], sub[col_name], style, label=eye, markersize=3, linewidth=1)
            ax.set_title(f"{net} {st}", fontsize=8)
            ax.grid(alpha=0.3); ax.legend(fontsize=6)
            ax.tick_params(axis="x", rotation=45, labelsize=7)
    fig.suptitle(f"Patient {pid} — Part 4: Caliber (3 net × 6 stat)", fontsize=13)
    plt.tight_layout(); plt.show()


# 🎯 Part 5. Knudtson — CRAE / CRVE / AVR (5)

**Zone B (0.5-1.0 DD from disc margin)** 안 상위 6 vessel segment 로 Knudtson iterative pairing:

- **CRAE** (Central Retinal Artery Equivalent): `sqrt(0.87 a₁² + 1.01 a₂² − 0.22 a₁a₂ − 10.76)` [μm]
- **CRVE** (Vein Equivalent): `sqrt(0.72 v₁² + 0.91 v₂² + 450.05)` [μm]
- **AVR** = CRAE / CRVE — 정상 ≈ **0.67**. 낮으면 고혈압/심혈관 위험 증가

함수: `compute_crae_crve_avr(av_mask, zone_b, mm_per_px, top_k=6)`.
출력: `CRAE_um, CRVE_um, AVR, n_artery_segments, n_vein_segments`


### Knudtson segment 시각화 -Zone B 안 탐지된 vessel segment

각 (환자, 좌우안) 의 fixed visit 기준:
- **Zone B 영역** (반투명 파랑)
- **Artery segment** (빨강, top-6 이 굵게 강조)
- **Vein segment** (파랑, top-6 이 굵게 강조)
- Segment 옆에 median diameter (μm) 표시


In [ ]:
# Zone B 안 vessel segment 시각화 + top-6 강조
from features.caliber import _segment_diameters
from scipy import ndimage as ndi
from scipy.ndimage import convolve as _conv
from skimage.morphology import skeletonize as _skel, remove_small_holes as _rsh, remove_small_objects as _rso

_K = np.array([[1,1,1],[1,0,1],[1,1,1]], dtype=np.uint8)

def _get_segments_in_zone(mask, zone, min_len=5):
    """Skeleton segments intersecting zone. Returns list of (segment_mask, median_diameter_px)."""
    m = mask.astype(bool)
    if not m.any(): return []
    m = _rsh(m, area_threshold=20)
    m = _rso(m, min_size=30)
    if not m.any(): return []
    dist = ndi.distance_transform_edt(m)
    skel = _skel(m)
    if not skel.any(): return []
    neigh = _conv(skel.astype(np.uint8), _K, mode="constant", cval=0)
    branches = skel & (neigh >= 3)
    segments = skel & ~branches
    if not segments.any(): return []
    labels, n = ndi.label(segments, structure=np.ones((3, 3), int))
    out = []
    for lbl in range(1, n + 1):
        seg = labels == lbl
        if (seg & zone).sum() < min_len: continue
        radii = dist[seg & zone]
        diameter = 2.0 * float(np.median(radii))
        out.append((seg & zone, diameter))
    return out

for (pid, eye), zones in zones_per_group.items():
    fixed = inventory[pid][eye][0]
    aligned = aligned_root / str(pid) / eye
    rgb_base = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    av_m = np.array(Image.open(aligned / "av" / f"{fixed['id']}.png"))
    zone_b = zones["zone_B"]
    mpp = zone_meta_df.query("patient==@pid & eye==@eye")["mm_per_px"].iloc[0]

    art_segs  = _get_segments_in_zone(np.isin(av_m, (1, 3)), zone_b)
    vein_segs = _get_segments_in_zone(np.isin(av_m, (2, 3)), zone_b)
    # top-6 (Knudtson) 선택
    art_top  = sorted(art_segs,  key=lambda x: -x[1])[:6]
    vein_top = sorted(vein_segs, key=lambda x: -x[1])[:6]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for ax, segs, top_segs, network, main_color, top_color in [
        (axes[0], art_segs,  art_top,  "Artery", [255, 100, 100], [255, 20, 20]),
        (axes[1], vein_segs, vein_top, "Vein",   [100, 130, 255], [30, 60, 220]),
    ]:
        disp = (rgb_base.astype(np.float32) * 0.5).astype(np.uint8)
        # Zone B 반투명 하이라이트
        disp[zone_b] = (disp[zone_b] * 0.7 + np.array([80, 150, 200]) * 0.3).astype(np.uint8)
        # 모든 segments (얇게)
        for seg, dia in segs:
            disp[seg] = main_color
        # top-6 강조 (dilate 로 두껍게)
        for seg, dia in top_segs:
            dilated = ndi.binary_dilation(seg, iterations=2)
            disp[dilated] = top_color
        ax.imshow(disp)
        # top-6 옆에 diameter 라벨
        for seg, dia in top_segs:
            ys, xs = np.where(seg)
            if len(xs) == 0: continue
            cy, cx = int(ys.mean()), int(xs.mean())
            dia_um = dia * mpp * 1000
            ax.text(cx, cy, f"{dia_um:.0f}", color="white", fontsize=8,
                    bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1))
        ax.set_title(f"{network} -{len(segs)} segments, top-6 강조 (μm)", fontsize=11)
        ax.axis("off")
    fig.suptitle(f"Patient {pid}/{eye} -Zone B 안 vessel segments (fixed visit)  "
                 f"[파란 영역 = Zone B, 빨강/파랑 = 모든 segments, 진한색 = Knudtson top-6]",
                 fontsize=11)
    plt.tight_layout(); plt.show()


In [ ]:
# CRAE / CRVE / AVR timeline
knudtson_metrics = ["CRAE_um", "CRVE_um", "AVR"]
fig, axes = plt.subplots(len(patient_ids), 3, figsize=(15, 4*len(patient_ids)), squeeze=False)
for row, pid in enumerate(patient_ids):
    for col, metric in enumerate(knudtson_metrics):
        ax = axes[row, col]
        for eye, style in [("L","-o"), ("R","-s")]:
            sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
            if len(sub) == 0 or metric not in sub: continue
            ax.plot(sub["date"], sub[metric], style, label=eye, markersize=4, linewidth=1.2)
        if metric == "AVR":
            ax.axhline(0.67, color="k", ls=":", alpha=0.5, label="normal ~0.67")
        ax.set_title(f"{pid} — {metric}", fontsize=11)
        if metric in {"CRAE_um","CRVE_um"}: ax.set_ylabel("μm")
        ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.suptitle("Part 5 — Knudtson CRAE / CRVE / AVR", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# n_segments (Knudtson 안 6개 이상 확보됐는지 QC)
fig, axes = plt.subplots(1, len(patient_ids), figsize=(6*len(patient_ids), 4), squeeze=False)
for col, pid in enumerate(patient_ids):
    ax = axes[0, col]
    for eye, style in [("L","-o"), ("R","-s")]:
        sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
        if len(sub) == 0: continue
        ax.plot(sub["date"], sub["n_artery_segments"], style, color="tab:red",
                label=f"{eye} artery", markersize=3, linewidth=0.9)
        ax.plot(sub["date"], sub["n_vein_segments"], style, color="tab:blue",
                label=f"{eye} vein", markersize=3, linewidth=0.9)
    ax.axhline(6, color="gray", ls="--", alpha=0.5, label="Knudtson top-6")
    ax.set_title(f"{pid} — n_segments in Zone B", fontsize=10)
    ax.set_ylabel("# segments"); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Part 5-QC — segment count (6+ = Knudtson valid)", fontsize=13)
plt.tight_layout(); plt.show()


# 🎨 Part 6. Color — arteriosclerosis marker (30)

**임상 배경**: 동맥경화 진행 시 vessel 벽이 두꺼워지고 색이 탈색됨.
- 정상 동맥: 밝은 빨강 (R 높음)
- Copper wire sign: 황갈색 톤 (early sclerosis)
- Silver wire sign: 은백색 (advanced sclerosis)

**아이디어**: Skeleton (vessel 중심선) 위에서 RGB 를 sampling → 색상 통계.
Sclerotic vessel 은 channel std 가 작아지며 **whiteness ↑, R/G ratio ↓**.

**추출 지표** (per network vessel/artery/vein × 9 stat + AV 비교 3):
- `mean_R, mean_G, mean_B` — 채널 평균
- `mean_intensity` — 회색값 평균
- `R_G_ratio, R_B_ratio` — 채색 강도
- `whiteness_mean, whiteness_p95` — 탈색도 (1 - channel_std/intensity)
- `AV_intensity_ratio, AV_whiteness_ratio, AV_RG_ratio` — artery ÷ vein


In [ ]:
# Color 시각화 -skeleton 값을 vessel 전체 폭으로 nearest-neighbor 로 전파
from scipy.ndimage import distance_transform_edt

def spread_scalar_to_vessel(vessel_bin, scalar_at_skel, skel_bool):
    """Skeleton pixel 값을 nearest-neighbor 로 vessel 전체에 전파 (viz 용)."""
    if not vessel_bin.any() or not skel_bool.any():
        return np.zeros_like(scalar_at_skel, dtype=np.float32)
    _, (ii, jj) = distance_transform_edt(~skel_bool, return_indices=True)
    dense = scalar_at_skel[ii, jj].astype(np.float32)
    dense[~vessel_bin] = 0.0
    return dense

for (pid, eye), zones in zones_per_group.items():
    fixed = inventory[pid][eye][0]
    aligned = aligned_root / str(pid) / eye
    rgb_base = np.array(Image.open(rgb_path / f"{fixed['id']}.png"))
    vs_m = np.array(Image.open(aligned / "vessels" / f"{fixed['id']}.png"))
    av_m = np.array(Image.open(aligned / "av"      / f"{fixed['id']}.png"))

    art_bin  = np.isin(av_m, (1, 3))
    vein_bin = np.isin(av_m, (2, 3))
    art_skel  = clean_and_skeletonize(art_bin)
    vein_skel = clean_and_skeletonize(vein_bin)

    r = rgb_base[..., 0].astype(np.float32)
    g = rgb_base[..., 1].astype(np.float32)
    b = rgb_base[..., 2].astype(np.float32)
    intensity = (r + g + b) / 3
    channel_std = np.std(np.stack([r,g,b], axis=0), axis=0)
    whiteness = 1 - channel_std / (intensity + 1e-6)
    rg_ratio  = r / (g + 1e-6)

    # skeleton 값을 vessel 전체 폭으로 전파해서 굵게
    a_int_dense = spread_scalar_to_vessel(art_bin,  np.where(art_skel,  intensity, 0), art_skel)
    v_int_dense = spread_scalar_to_vessel(vein_bin, np.where(vein_skel, intensity, 0), vein_skel)
    a_wh_dense  = spread_scalar_to_vessel(art_bin,  np.where(art_skel,  whiteness, 0), art_skel)
    v_wh_dense  = spread_scalar_to_vessel(vein_bin, np.where(vein_skel, whiteness, 0), vein_skel)
    a_rg_dense  = spread_scalar_to_vessel(art_bin,  np.where(art_skel,  rg_ratio,  0), art_skel)
    v_rg_dense  = spread_scalar_to_vessel(vein_bin, np.where(vein_skel, rg_ratio,  0), vein_skel)

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Row 0: base RGB + artery/vein whiteness (sclerosis marker)
    axes[0, 0].imshow(rgb_base)
    axes[0, 0].set_title(f"Base RGB (Patient {pid}/{eye})", fontsize=11)
    axes[0, 0].axis("off")

    disp = (rgb_base.astype(np.float32) * 0.45).astype(np.uint8)
    axes[0, 1].imshow(disp)
    im = axes[0, 1].imshow(np.ma.masked_where(a_wh_dense == 0, a_wh_dense),
                           cmap="hot", alpha=1.0, vmin=0.3, vmax=0.9)
    plt.colorbar(im, ax=axes[0, 1], fraction=0.04, label="whiteness")
    axes[0, 1].set_title("Artery whiteness (↑ = sclerosis)", fontsize=11)
    axes[0, 1].axis("off")

    disp = (rgb_base.astype(np.float32) * 0.45).astype(np.uint8)
    im = axes[0, 2].imshow(disp)
    im = axes[0, 2].imshow(np.ma.masked_where(v_wh_dense == 0, v_wh_dense),
                           cmap="hot", alpha=1.0, vmin=0.3, vmax=0.9)
    plt.colorbar(im, ax=axes[0, 2], fraction=0.04, label="whiteness")
    axes[0, 2].set_title("Vein whiteness (baseline)", fontsize=11)
    axes[0, 2].axis("off")

    # Row 1: intensity + R/G ratio
    disp = (rgb_base.astype(np.float32) * 0.45).astype(np.uint8)
    axes[1, 0].imshow(disp)
    im = axes[1, 0].imshow(np.ma.masked_where(a_int_dense == 0, a_int_dense),
                           cmap="Reds", alpha=1.0, vmin=40, vmax=200)
    axes[1, 0].imshow(np.ma.masked_where(v_int_dense == 0, v_int_dense),
                      cmap="Blues", alpha=1.0, vmin=40, vmax=200)
    axes[1, 0].set_title("Intensity (R=artery Reds, B=vein Blues)", fontsize=11)
    axes[1, 0].axis("off")

    disp = (rgb_base.astype(np.float32) * 0.45).astype(np.uint8)
    axes[1, 1].imshow(disp)
    im = axes[1, 1].imshow(np.ma.masked_where(a_rg_dense == 0, a_rg_dense),
                           cmap="autumn", alpha=1.0, vmin=1.0, vmax=3.0)
    plt.colorbar(im, ax=axes[1, 1], fraction=0.04, label="R/G")
    axes[1, 1].set_title("Artery R/G ratio (↓ = sclerosis)", fontsize=11)
    axes[1, 1].axis("off")

    disp = (rgb_base.astype(np.float32) * 0.45).astype(np.uint8)
    axes[1, 2].imshow(disp)
    im = axes[1, 2].imshow(np.ma.masked_where(v_rg_dense == 0, v_rg_dense),
                           cmap="autumn", alpha=1.0, vmin=1.0, vmax=3.0)
    plt.colorbar(im, ax=axes[1, 2], fraction=0.04, label="R/G")
    axes[1, 2].set_title("Vein R/G ratio", fontsize=11)
    axes[1, 2].axis("off")

    fig.suptitle(f"Patient {pid}/{eye} -Skeleton color (spread to vessel width)", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# Color feature timeline — 3 network × 4 key stat (intensity, R_G_ratio, whiteness_mean, whiteness_p95)
color_key_stats = ["mean_intensity", "R_G_ratio", "whiteness_mean", "whiteness_p95"]
networks = ["vessel", "artery", "vein"]
for pid in patient_ids:
    fig, axes = plt.subplots(3, len(color_key_stats),
                             figsize=(4*len(color_key_stats), 8), squeeze=False, sharex=True)
    for r, net in enumerate(networks):
        for c, st in enumerate(color_key_stats):
            ax = axes[r, c]
            col_name = f"whole_{net}_{st}"
            for eye, style in [("L","-o"),("R","-s")]:
                sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
                if len(sub) == 0 or col_name not in sub: continue
                ax.plot(sub["date"], sub[col_name], style, label=eye, markersize=3, linewidth=1)
            ax.set_title(f"{net} {st}", fontsize=9)
            ax.grid(alpha=0.3); ax.legend(fontsize=7)
            ax.tick_params(axis="x", rotation=45, labelsize=7)
    fig.suptitle(f"Patient {pid} — Part 6: Color at skeleton (3 net × 4 key stat)", fontsize=13)
    plt.tight_layout(); plt.show()

# AV 비교 지표 timeline
fig, axes = plt.subplots(1, 3, figsize=(15, 4), squeeze=False)
for c, met in enumerate(["AV_intensity_ratio","AV_whiteness_ratio","AV_RG_ratio"]):
    ax = axes[0, c]
    col_name = f"whole_{met}"
    for pid in patient_ids:
        for eye, style in [("L","-o"),("R","-s")]:
            sub = features_zones[(features_zones["patient"]==pid) & (features_zones["eye"]==eye)]
            if len(sub) == 0 or col_name not in sub: continue
            ax.plot(sub["date"], sub[col_name], style, label=f"{pid} {eye}", markersize=3, linewidth=1)
    ax.axhline(1, color="k", ls=":", alpha=0.5, label="A=V")
    ax.set_title(met); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.suptitle("Part 6b: AV color ratios — sclerosis marker (artery ↓ 시 A/V ratio ↓)", fontsize=13)
plt.tight_layout(); plt.show()
